# 33 AASM Features + Classical ML, under Leave-One-Patient-Out

The same 89-patient LOPO protocol as the deep-learning notebooks, the same metrics,
the same figures and CSVs — but the model is a **classical classifier on the 33 AASM
rule features** instead of a CNN+BiGRU on raw waveforms.

| | CNN + BiGRU notebook | this notebook |
|---|---|---|
| Input per epoch | 5×960 @32 Hz + 3×120 @4 Hz waveforms | **33 scalars** |
| Temporal context | BiGRU over 40 epochs (20 min) | none by default (see `CONTEXT_LAGS`) |
| Models | 1 network, 486 K parameters | **5 classifiers** |
| Runtime | ~6 h on a 4090 | **~20–40 min on CPU** |

### Why this is worth running

It is the **feature-sufficiency experiment**. The deep model gets raw signal *and* the
33 features; this one gets only the features. The gap between the two is the value of
learned representation over hand-crafted rules on this cohort — a question the
without-features ablation answers from the other direction. Together the three runs
bracket it completely.

It is also fast enough to be a genuine baseline. A reviewer who asks "did you try a
random forest on the rule features?" gets a 89-fold LOPO answer, not a hand-wave.

### Where the 33 features come from

They are **already in your cache**. `apnea_cache_resampled/*.npz` carries a `feat`
array of shape `(n_epochs, 33)`, computed at cache-build time from the **raw 200 Hz
signal before resampling and normalization** — which is the correct place to compute
them, since they encode physical amplitude relationships.

This notebook uses those cached values by default. A recomputation path from the
32 Hz / 4 Hz signals is included for caches that lack `feat`, with an explicit warning
about which features survive that route and which do not (see Cell 3).

### The five models

`LogisticRegression` · `RandomForest` · `HistGradientBoosting` · `XGBoost` · `SVM_RBF`

All five get the same folds, the same class weighting, and a threshold tuned on
validation only. Two ensembles (soft vote, logistic stacker) come free from the stored
validation probabilities.

## Cell 1 — Imports and configuration

### Output layout

```
33features_ml_results/
├── folds/{model}/{pid}.npz        cached per-fold predictions (resumable)
├── csv/                           every table
├── figures/                       every plot
└── models/                        one fitted pipeline per model (fold 0)
```

### "Use the best loss function from above"

Classical classifiers do not have your loss arms — there is no focal term in a random
forest. What *does* carry over is the **class-weighting half** of the 2×2:

| Loss arm | Classical analogue |
|---|---|
| `bce` | `class_weight = None` |
| `weighted_bce` | `class_weight = "balanced"` |
| `focal` | no direct analogue (down-weights easy examples) |
| `weighted_focal` | `class_weight = "balanced"` |

`BEST_LOSS_ARM` below performs that mapping. Set it to whichever arm won your ablation
on severity κ<sub>w</sub>; the notebook translates and prints what it chose. Since focal
has no tree analogue, an arm containing "focal" maps on its weighting component only —
stated in the printout so the paper can describe it accurately.

### Context features

`CONTEXT_LAGS = ()` by default, so each epoch is classified from **its own 33 features
alone** — exactly "33 AASM features per window" as specified. Setting it to `(-2,-1,1,2)`
copies neighbouring epochs' features into each row, which is the classical-ML substitute
for the BiGRU's context and typically the single largest lever on F1. It is left off so
the headline result answers the question as posed; turn it on for a second run if you
want the comparison.

In [ ]:
# =========================================================================
# CELL 1 — imports and configuration
# =========================================================================
import os, re, gc, glob, json, time, warnings, sys, subprocess, importlib.util
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

if importlib.util.find_spec("xgboost") is None:
    _c = [sys.executable, "-m", "pip", "install", "-q", "xgboost"]
    try:    subprocess.check_call(_c)
    except Exception: subprocess.call(_c + ["--break-system-packages"])
HAS_XGB = importlib.util.find_spec("xgboost") is not None

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.inspection import permutation_importance
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, roc_auc_score, average_precision_score,
                             precision_recall_curve, matthews_corrcoef,
                             cohen_kappa_score, brier_score_loss, roc_curve)
from scipy.stats import pearsonr, spearmanr, wilcoxon

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 60)

# ---- paths --------------------------------------------------------------
PROJECT_DIR = "/home/nishant/apneastudyproject"
CACHE_DIR   = os.environ.get("APNEA_CACHE",  os.path.join(PROJECT_DIR, "apnea_cache_resampled"))
OUT_DIR     = os.environ.get("APNEA_RESULT", os.path.join(PROJECT_DIR, "33features_ml_results"))
FOLD_DIR    = os.path.join(OUT_DIR, "folds")
CSV_DIR     = os.path.join(OUT_DIR, "csv")
FIG_DIR     = os.path.join(OUT_DIR, "figures")
MODEL_DIR   = os.path.join(OUT_DIR, "models")
for d in (OUT_DIR, FOLD_DIR, CSV_DIR, FIG_DIR, MODEL_DIR): os.makedirs(d, exist_ok=True)

# ---- protocol (identical to the LOPO notebooks) -------------------------
SEED, VAL_K = 42, 12
WINDOW_SEC, CHUNK_EPOCHS = 30, 40      # CHUNK_EPOCHS only gates the min-length check
EPH_BINS = [5.0, 15.0, 30.0]
SEVERITY_NAMES = ["normal", "mild", "moderate", "severe"]
N_CALIB_BINS = 10

# ---- "best loss" -> class weighting -------------------------------------
BEST_LOSS_ARM = "weighted_focal"       # <- set to the arm that won your ablation
CLASS_WEIGHT  = "balanced" if BEST_LOSS_ARM.startswith("weighted") else None

# ---- features -----------------------------------------------------------
USE_CACHED_FEATURES = True   # True: use cache['feat'] (exact, from raw 200 Hz)
                             # False: recompute from the 32/4 Hz signals (see Cell 3)
CONTEXT_LAGS = ()            # () = 33 features per window, as specified.
                             # (-2,-1,1,2) adds neighbouring epochs -> 165 columns
CONTEXT_ROLL = 0             # >0 adds (x - rolling median) deltas over that many epochs

# ---- models -------------------------------------------------------------
MODELS_TO_RUN = ["LogisticRegression", "RandomForest", "HistGradientBoosting",
                 "XGBoost", "SVM_RBF"]
BASELINE_MODEL = "LogisticRegression"      # paired tests compare against this
SUBSAMPLE_SVM = 12000        # kernel SVM is O(n^2); tree models get everything
N_JOBS = -1
MAX_FOLDS = None             # 3 for a smoke run, None for all 89
RESUME = True

MODEL_COLOR = {"LogisticRegression": "#4c78a8", "RandomForest": "#59a14f",
               "HistGradientBoosting": "#e1913c", "XGBoost": "#d1495b",
               "SVM_RBF": "#8e7cc3", "SoftVote": "#7a869a", "Stacked_LR": "#333333"}

print(f"cache  {CACHE_DIR}")
print(f"output {OUT_DIR}")
print(f"best loss arm '{BEST_LOSS_ARM}' -> class_weight={CLASS_WEIGHT}"
      + ("  (focal has no tree analogue; weighting component only)"
         if "focal" in BEST_LOSS_ARM else ""))
print(f"models {MODELS_TO_RUN}  | xgboost={HAS_XGB}")
print(f"context lags {CONTEXT_LAGS or 'none (33 features per window)'}")

## Cell 2 — Load the cache

Verbatim from the LOPO notebook, so the cohort, the exclusions, the
`events_per_hour` definition and the severity bands are identical across all four
studies. Patients with fewer than `CHUNK_EPOCHS` epochs or zero positive epochs are
dropped, exactly as before.

In [ ]:
# =========================================================================
# CELL 2 — load the pre-built cache
# =========================================================================
def boundary_targets(y):
    """onset[t]=1 where an event starts, offset[t]=1 where it ends."""
    y = np.asarray(y).astype(np.int8)
    onset, offset = np.zeros(len(y), np.float32), np.zeros(len(y), np.float32)
    if len(y) == 0:
        return onset, offset
    prev = np.concatenate(([0], y[:-1]))
    nxt = np.concatenate((y[1:], [0]))
    onset[(y == 1) & (prev == 0)] = 1.0
    offset[(y == 1) & (nxt == 0)] = 1.0
    return onset, offset


def load_cache(cache_dir):
    files = sorted(glob.glob(os.path.join(cache_dir, "*.npz")))
    if not files:
        raise FileNotFoundError(f"no .npz files in {cache_dir}")
    cache = {}
    for f in files:
        m = re.search(r"(SN\d+)", os.path.basename(f))
        if not m:
            continue
        pid = m.group(1)
        with np.load(f, allow_pickle=True) as z:
            if not {"resp", "slow", "y"} <= set(z.files):
                print(f"  [skip] {pid}: missing resp/slow/y (has {list(z.files)})")
                continue
            resp, slow = z["resp"], z["slow"]
            y = np.asarray(z["y"]).astype(np.int8)
            n = min(len(y), resp.shape[0], slow.shape[0])
            d = dict(resp=resp[:n].copy(), slow=slow[:n].copy(), y=y[:n])
            d["feat"] = (np.asarray(z["feat"], np.float32)[:n] if "feat" in z.files
                         else np.zeros((n, 0), np.float32))
            if "onset" in z.files and "offset" in z.files:
                d["onset"] = np.asarray(z["onset"], np.float32)[:n]
                d["offset"] = np.asarray(z["offset"], np.float32)[:n]
            else:
                d["onset"], d["offset"] = boundary_targets(d["y"])
            d["mask_resp"] = (np.asarray(z["mask_resp"], np.float32) if "mask_resp" in z.files
                              else np.ones(resp.shape[1], np.float32))
            d["mask_slow"] = (np.asarray(z["mask_slow"], np.float32) if "mask_slow" in z.files
                              else np.ones(slow.shape[1], np.float32))
        if n >= CHUNK_EPOCHS and y[:n].sum() > 0:
            cache[pid] = d
        else:
            print(f"  [skip] {pid}: {n} epochs, {int(y[:n].sum())} positive")
    return cache


def find_events(seq):
    b = np.asarray(seq).astype(int)
    if b.size == 0:
        return []
    dd = np.diff(np.concatenate(([0], b, [0])))
    return list(zip(np.flatnonzero(dd == 1).tolist(),
                    (np.flatnonzero(dd == -1) - 1).tolist()))


def events_per_hour(y):
    """Events per hour of RECORDING (no sleep staging in this dataset)."""
    return len(find_events(y)) / max(len(y) * WINDOW_SEC / 3600.0, 1e-6)


def severity_of(a):
    return int(np.searchsorted(EPH_BINS, a, side="right"))


CACHE = load_cache(CACHE_DIR)
PIDS = sorted(CACHE, key=lambda p: int(p[2:]))

# layout is READ FROM THE DATA, not assumed
_d0 = CACHE[PIDS[0]]
N_RESP, RESP_SPE = _d0["resp"].shape[1], _d0["resp"].shape[2]
N_SLOW, SLOW_SPE = _d0["slow"].shape[1], _d0["slow"].shape[2]
N_FEAT = _d0["feat"].shape[1]
N_MASK = N_RESP + N_SLOW

COHORT = pd.DataFrame([dict(patient=p, n_epochs=len(CACHE[p]["y"]),
                            apnea_ratio=float(CACHE[p]["y"].mean()),
                            events_per_hour=events_per_hour(CACHE[p]["y"]),
                            severity=SEVERITY_NAMES[severity_of(events_per_hour(CACHE[p]["y"]))])
                       for p in PIDS])
COHORT.to_csv(os.path.join(OUT_DIR, "cohort.csv"), index=False)

n_ep = int(COHORT.n_epochs.sum())
print(f"\n{len(PIDS)} patients | {n_ep:,} epochs ({n_ep*WINDOW_SEC/3600:.0f} h) | "
      f"apnea {100*sum(CACHE[p]['y'].sum() for p in PIDS)/n_ep:.1f}%")
print(f"resp ({N_RESP}, {RESP_SPE}) @ {RESP_SPE/WINDOW_SEC:.0f} Hz | "
      f"slow ({N_SLOW}, {SLOW_SPE}) @ {SLOW_SPE/WINDOW_SEC:.0f} Hz | {N_FEAT} features")
print(COHORT.severity.value_counts().reindex(SEVERITY_NAMES).fillna(0).astype(int).to_string())

## Cell 3 — The 33 AASM features

### Default path: use what is cached

The cached `feat` array was computed from the **raw 200 Hz signal before resampling and
normalization**, which is where these features belong — they encode physical amplitude
relationships that normalization would destroy. Cell 2 reports the feature count; if it
says 33, nothing needs recomputing.

### Fallback path: recompute from the 32 Hz / 4 Hz signals

`recompute_aasm()` reconstructs the features from the resampled cache. It is provided
for completeness, but **it is not equivalent**, and the differences are worth stating
plainly:

| Feature group | Survives recomputation? | Why |
|---|---|---|
| amplitude ratios (1–5, ×4 channels) | **Yes, exactly** | every one is a ratio, so the per-channel normalization constant cancels in the division |
| paradox correlation (17) | **Yes, exactly** | Pearson r is scale- and shift-invariant |
| thorax/abd ratio (18) | **Approximately** | the two channels were normalized by *different* IQRs, so the ratio is scaled by a constant |
| SpO₂ (6–12) | **Yes** | the affine scaling `(x−90)/10` is invertible: `x = 10s + 90` recovers true percentages |
| pulse (13–16) | **In normalized units** | pulse was median/IQR-normalized and the IQR is not stored, so `pulse_std` is no longer in bpm |
| all | **Resolution** | the 1 Hz envelope now comes from 32 Hz rather than 200 Hz samples, so peak-to-peak is slightly attenuated |

Set `USE_CACHED_FEATURES = False` only if your cache genuinely lacks `feat`, and say so
in the methods section if you do.

In [ ]:
# =========================================================================
# CELL 3 — the 33 AASM features
# =========================================================================
from scipy.ndimage import median_filter, maximum_filter1d

FEAT_RESP_KEYS = ["flow", "chest", "abd", "flow2"]
RESP_KEYS = ["flow", "chest", "abd", "pleth", "flow2"]
SLOW_KEYS = ["spo2", "pulse", "ecg_env"]

def aasm_feature_names():
    n = []
    for k in FEAT_RESP_KEYS:
        n += [f"{k}_amp_ratio", f"{k}_min_ratio", f"{k}_sec_below_70",
              f"{k}_sec_below_50", f"{k}_longest_low_run"]
    n += ["spo2_min", "spo2_mean", "spo2_range", "spo2_desat",
          "spo2_desat_ge3", "spo2_below_90", "spo2_slope"]
    n += ["pulse_delta_base", "pulse_std", "pulse_range", "pulse_max_jump"]
    n += ["paradox_corr", "thorax_abd_ratio"]
    return n

AASM_NAMES = aasm_feature_names()
assert len(AASM_NAMES) == 33

def _ratio(a, b, eps=1e-6):
    return np.asarray(a, np.float64) / (np.abs(np.asarray(b, np.float64)) + eps)

def _longest_run(mask):
    n, T = mask.shape
    run = np.zeros(n, np.int32); best = np.zeros(n, np.int32)
    for t in range(T):
        run = np.where(mask[:, t], run + 1, 0)
        best = np.maximum(best, run)
    return best.astype(np.float32)

def recompute_aasm(d):
    """Rebuild the 33 features from the RESAMPLED cache. See the markdown for caveats."""
    resp = np.asarray(d["resp"], np.float32)       # (n, 5, 960) @ 32 Hz
    slow = np.asarray(d["slow"], np.float32)       # (n, 3, 120) @ 4 Hz
    n_ep = resp.shape[0]
    rfs = resp.shape[2] // WINDOW_SEC              # 32
    sfs = slow.shape[2] // WINDOW_SEC              # 4
    out, env = {}, {}

    for i, key in enumerate(RESP_KEYS):
        if key not in FEAT_RESP_KEYS:
            continue
        amp = (resp[:, i].reshape(n_ep, WINDOW_SEC, rfs).max(2) -
               resp[:, i].reshape(n_ep, WINDOW_SEC, rfs).min(2))     # 1 Hz envelope
        env[key] = amp
        ep_amp = amp.mean(1)
        base = median_filter(ep_amp.astype(np.float64), size=9, mode="nearest")
        r = _ratio(amp, base[:, None])
        out[f"{key}_amp_ratio"] = _ratio(ep_amp, base)
        out[f"{key}_min_ratio"] = _ratio(amp.min(1), base)
        out[f"{key}_sec_below_70"] = (r < 0.70).mean(1)
        out[f"{key}_sec_below_50"] = (r < 0.50).mean(1)
        out[f"{key}_longest_low_run"] = _longest_run(r < 0.50)

    # SpO2: invert the affine scaling to recover true percentages
    sp = slow[:, SLOW_KEYS.index("spo2")].reshape(n_ep, WINDOW_SEC, sfs).mean(2)
    sp = np.clip(sp * 10.0 + 90.0, 50.0, 100.0)
    ep_min, ep_max = sp.min(1), sp.max(1)
    base = maximum_filter1d(ep_max.astype(np.float64), size=9, mode="nearest", origin=4)
    out["spo2_min"] = ep_min
    out["spo2_mean"] = sp.mean(1)
    out["spo2_range"] = ep_max - ep_min
    out["spo2_desat"] = base - ep_min
    out["spo2_desat_ge3"] = (base - ep_min >= 3.0).astype(np.float64)
    out["spo2_below_90"] = (sp < 90.0).mean(1)
    out["spo2_slope"] = sp[:, -1] - sp[:, 0]

    p = slow[:, SLOW_KEYS.index("pulse")].reshape(n_ep, WINDOW_SEC, sfs).mean(2)
    ep = p.mean(1)
    out["pulse_delta_base"] = ep - median_filter(ep.astype(np.float64), size=9, mode="nearest")
    out["pulse_std"] = p.std(1)
    out["pulse_range"] = p.max(1) - p.min(1)
    out["pulse_max_jump"] = np.abs(np.diff(p, axis=1)).max(1)

    if "chest" in env and "abd" in env:
        a = env["chest"] - env["chest"].mean(1, keepdims=True)
        b = env["abd"] - env["abd"].mean(1, keepdims=True)
        out["paradox_corr"] = (a * b).sum(1) / (np.sqrt((a**2).sum(1) * (b**2).sum(1)) + 1e-9)
        out["thorax_abd_ratio"] = _ratio(env["chest"].mean(1), env["abd"].mean(1))
    else:
        out["paradox_corr"] = np.zeros(n_ep); out["thorax_abd_ratio"] = np.zeros(n_ep)

    X = np.zeros((n_ep, 33), np.float32)
    for j, nm in enumerate(AASM_NAMES):
        if nm in out:
            X[:, j] = np.nan_to_num(np.asarray(out[nm], np.float32), posinf=0, neginf=0)
    return X


def add_context(X, lags=CONTEXT_LAGS, roll=CONTEXT_ROLL):
    """Lag/lead copies and rolling-baseline deltas, per patient (edge-repeat padded)."""
    blocks, names = [X], list(AASM_NAMES)
    if roll and roll > 1:
        base = median_filter(X, size=(roll, 1), mode="nearest")
        blocks.append(X - base); names += [f"{n}__dbase" for n in AASM_NAMES]
    for lg in lags:
        k = abs(lg)
        sh = (np.concatenate([np.repeat(X[:1], k, 0), X[:-k]], 0) if lg > 0
              else np.concatenate([X[k:], np.repeat(X[-1:], k, 0)], 0))
        blocks.append(sh); names += [f"{n}__lag{lg:+d}" for n in AASM_NAMES]
    return np.concatenate(blocks, 1).astype(np.float32), names


# ---- build the feature matrix for every patient -------------------------
FEATURE_NAMES = None
n_recomputed = 0
for pid, d in CACHE.items():
    F = d["feat"] if (USE_CACHED_FEATURES and d["feat"].shape[1] == 33) else None
    if F is None:
        F = recompute_aasm(d); n_recomputed += 1
    Xc, names = add_context(np.asarray(F, np.float32))
    d["X"] = np.nan_to_num(Xc, nan=0.0, posinf=0.0, neginf=0.0)
    if FEATURE_NAMES is None:
        FEATURE_NAMES = names
    elif names != FEATURE_NAMES:
        raise RuntimeError(f"{pid}: feature names differ")

D = len(FEATURE_NAMES)
print(f"feature matrix: {D} columns "
      f"({'cached' if n_recomputed == 0 else f'{n_recomputed} patients recomputed'})")
if n_recomputed:
    print("  [!] recomputed from the resampled cache -- pulse features are in normalized")
    print("      units and thorax_abd_ratio is scaled. See the markdown above.")
print(f"total rows: {sum(len(CACHE[p]['y']) for p in PIDS):,}")
print("features:", ", ".join(FEATURE_NAMES[:5]), "...", FEATURE_NAMES[-1])

## Cell 4 — Folds, metrics, and events-per-hour agreement

`make_fold` is copied from the LOPO notebooks and is **deterministic and
model-independent**: the held-out patient plus 12 severity-ranked validation patients,
76 for training. Every model therefore sees identical folds, which is what makes the
per-patient results **paired** and licenses the Wilcoxon tests in Cell 8.

`patient_metrics` carries the F1 fix: `NaN` only when the *labels* make a metric
undefined (a patient with no positive epochs), `0.0` when the model simply failed
(predicted nothing, or nothing correct). Recording a real failure as `NaN` silently
deletes the worst patients from the average.

`eph_agreement` reports **both** R-squareds. Squared Pearson correlation measures fit to
a best-fit *line* and is blind to systematic offset; the coefficient of determination
measures fit to the *identity* line and penalises bias. With a non-zero bias they
diverge, and the determination version is the one to report.

In [ ]:
# =========================================================================
# CELL 4a — folds
# =========================================================================
def make_fold(test_pid):
    """Deterministic and model-independent -> all models are exactly paired."""
    others = [p for p in PIDS if p != test_pid]
    ranks = np.argsort([events_per_hour(CACHE[p]["y"]) for p in others], kind="stable")
    ordered = [others[i] for i in ranks]
    vi = sorted(set(np.linspace(0, len(ordered) - 1, VAL_K).round().astype(int).tolist()))
    return ([p for i, p in enumerate(ordered) if i not in set(vi)],
            [ordered[i] for i in vi], [test_pid])


def assemble(pids):
    X = np.concatenate([CACHE[p]["X"] for p in pids]).astype(np.float32)
    y = np.concatenate([CACHE[p]["y"] for p in pids]).astype(np.int8)
    return X, y


def best_f1_threshold(labels, probs):
    """F1-optimal cut. Only ever called on VALIDATION."""
    labels, probs = np.asarray(labels), np.asarray(probs)
    if labels.size == 0 or len(np.unique(labels)) < 2:
        return 0.5, 0.0
    prec, rec, thr = precision_recall_curve(labels, probs)
    if len(thr) == 0:
        return 0.5, 0.0
    f1s = 2 * prec * rec / (prec + rec + 1e-9)
    i = int(np.nanargmax(f1s[:-1]))
    return float(thr[i]), float(f1s[i])

print("folds ready |", len(PIDS), "patients ->", len(PIDS), "LOPO folds")

In [ ]:
# =========================================================================
# CELL 4b — per-patient metrics
# =========================================================================
def expected_calibration_error(labels, probs, n_bins=N_CALIB_BINS):
    labels, probs = np.asarray(labels).astype(int), np.asarray(probs, float)
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece, N = 0.0, len(probs)
    for i in range(n_bins):
        hi = probs <= edges[i + 1] if i == n_bins - 1 else probs < edges[i + 1]
        m = (probs >= edges[i]) & hi
        if m.any():
            ece += (m.sum() / N) * abs(labels[m].mean() - probs[m].mean())
    return float(ece)


def patient_metrics(labels, probs, thr):
    """NaN only when the LABELS make a metric undefined; 0.0 when the model failed."""
    labels = np.asarray(labels).astype(int)
    pred = (np.asarray(probs) >= thr).astype(int)
    has_pos, has_neg = labels.sum() > 0, (labels == 0).sum() > 0
    two = has_pos and has_neg
    tn, fp, fn, tp = confusion_matrix(labels, pred, labels=[0, 1]).ravel()

    sens = (tp / (tp + fn)) if has_pos else np.nan
    spec = (tn / (tn + fp)) if has_neg else np.nan
    # precision: 0/0 with true positives present is a real failure -> 0.0
    prec = (tp / (tp + fp)) if (tp + fp) else (0.0 if has_pos else np.nan)
    if not has_pos:
        f1 = np.nan
    elif (prec + sens) == 0:
        f1 = 0.0                       # <-- the fix
    else:
        f1 = 2 * prec * sens / (prec + sens)

    return dict(
        n_epochs=len(labels), n_apnea=int(labels.sum()),
        prevalence=float(labels.mean()), threshold=float(thr),
        accuracy=accuracy_score(labels, pred),
        precision=prec, recall=sens, sensitivity=sens, specificity=spec,
        npv=(tn / (tn + fn)) if (tn + fn) else np.nan,
        f1=f1,
        balanced_accuracy=(sens + spec) / 2 if np.isfinite(sens) and np.isfinite(spec) else np.nan,
        mcc=matthews_corrcoef(labels, pred) if two else np.nan,
        cohen_kappa=cohen_kappa_score(labels, pred) if two else np.nan,
        roc_auc=roc_auc_score(labels, probs) if two else np.nan,
        pr_auc=average_precision_score(labels, probs) if two else np.nan,
        brier=brier_score_loss(labels, np.clip(probs, 0, 1)),
        ece=expected_calibration_error(labels, probs),
        tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp),
        true_eph=events_per_hour(labels), pred_eph=events_per_hour(pred),
        severity=SEVERITY_NAMES[severity_of(events_per_hour(labels))],
        n_true_events=len(find_events(labels)), n_pred_events=len(find_events(pred)))


METRIC_COLS = ["accuracy", "precision", "recall", "sensitivity", "specificity", "npv",
               "f1", "balanced_accuracy", "mcc", "cohen_kappa", "roc_auc", "pr_auc",
               "brier", "ece"]
print("metrics ready |", len(METRIC_COLS), "averaged metrics")

In [ ]:
# =========================================================================
# CELL 4c — events-per-hour agreement
# =========================================================================
def eph_agreement(df):
    """Cohort-level agreement between true and predicted events per hour."""
    d = df[["true_eph", "pred_eph"]].dropna()
    out = dict(n=len(d))
    if len(d) < 3:
        return {**out, **{k: np.nan for k in
                ("eph_pearson_r", "eph_r2_pearson", "eph_r2_determination", "eph_spearman",
                 "eph_mae", "eph_rmse", "eph_bias", "eph_loa_lo", "eph_loa_hi",
                 "severity_acc", "severity_kappa_w")}}
    t, p = d.true_eph.values, d.pred_eph.values
    diff = p - t
    r = float(pearsonr(t, p)[0])
    ss_res = float(((p - t) ** 2).sum())
    ss_tot = float(((t - t.mean()) ** 2).sum())
    ts = np.array([severity_of(v) for v in t])
    ps = np.array([severity_of(v) for v in p])
    out.update(eph_pearson_r=r,
               eph_r2_pearson=r ** 2,                                  # line-of-best-fit
               eph_r2_determination=1 - ss_res / max(ss_tot, 1e-9),    # identity line
               eph_spearman=float(spearmanr(t, p)[0]),
               eph_mae=float(np.abs(diff).mean()),
               eph_rmse=float(np.sqrt((diff ** 2).mean())),
               eph_bias=float(diff.mean()),
               eph_loa_lo=float(diff.mean() - 1.96 * diff.std(ddof=1)),
               eph_loa_hi=float(diff.mean() + 1.96 * diff.std(ddof=1)),
               severity_acc=float((ts == ps).mean()),
               severity_kappa_w=float(cohen_kappa_score(ts, ps, weights="quadratic"))
                                 if len(np.unique(ts)) > 1 else np.nan)
    return out

print("events-per-hour agreement ready")

## Cell 5 — The five models

Two preprocessing regimes, for a real reason.

**Trees are scale-invariant.** A tree splits on `x < 3.7`; multiplying `x` by 100 just
moves the threshold. `RandomForest`, `HistGradientBoosting` and `XGBoost` get the raw
matrix — scaling them would cost time and add float noise for no benefit.

**Everything else needs scaling.** `LogisticRegression` and `SVM_RBF` compute dot
products and distances, so a feature ranging over hundreds (`pulse_range`) would drown
one ranging over fractions (`spo2_desat_ge3`). Both get `StandardScaler`, and both are
clipped to ±10 afterwards so a single outlier epoch cannot dominate.

### Class weighting

All five take `class_weight="balanced"` (or `scale_pos_weight` for XGBoost, which is the
same idea expressed differently), derived from `BEST_LOSS_ARM` in Cell 1. **One
correction only** — never combine weighting with resampling; the corrections multiply
and massively over-shoot the positive class.

### The SVM subsample

Kernel SVM is O(n²) in training samples. A fold has ~72,000 training rows, which would
mean ~5 billion kernel entries. It gets a **stratified** subsample of 12,000 — all
positives up to half the budget, negatives for the rest — which keeps roughly the
training prevalence rather than forcing 50/50 (that would be the double-correction error
again). Validation and test are always scored in full.

`probability=True` runs internal Platt scaling so the SVM emits calibrated
probabilities; without it Brier score and ECE would be undefined and the reliability
diagram would be meaningless.

In [ ]:
# =========================================================================
# CELL 5 — the model zoo
# =========================================================================
class Clip(StandardScaler):
    """StandardScaler that also clips, so one outlier epoch cannot dominate."""
    def transform(self, X, copy=None):
        return np.clip(super().transform(X, copy=copy), -10.0, 10.0)

def pre_scaled(): return [("imp", SimpleImputer(strategy="median")), ("sc", Clip())]
def pre_raw():    return [("imp", SimpleImputer(strategy="median"))]


def build_models():
    Z = {}
    cw = CLASS_WEIGHT
    Z["LogisticRegression"] = dict(subsample=None, note="linear baseline on 33 features",
        make=lambda: Pipeline(pre_scaled() + [("clf", LogisticRegression(
            max_iter=3000, C=1.0, class_weight=cw, n_jobs=N_JOBS))]))
    Z["RandomForest"] = dict(subsample=None, note="bagged decorrelated deep trees",
        make=lambda: Pipeline(pre_raw() + [("clf", RandomForestClassifier(
            n_estimators=400, min_samples_leaf=5, max_features="sqrt",
            class_weight=cw, n_jobs=N_JOBS, random_state=SEED))]))
    Z["HistGradientBoosting"] = dict(subsample=None, note="binned boosting, fast",
        make=lambda: Pipeline(pre_raw() + [("clf", _hgb(cw))]))
    if HAS_XGB:
        from xgboost import XGBClassifier
        Z["XGBoost"] = dict(subsample=None, pos_weight=True,
            note="second-order gradients + regularization",
            make=lambda: Pipeline(pre_raw() + [("clf", XGBClassifier(
                n_estimators=500, max_depth=6, learning_rate=0.06, subsample=0.8,
                colsample_bytree=0.8, min_child_weight=5, reg_lambda=1.0,
                tree_method="hist", eval_metric="logloss",
                n_jobs=N_JOBS, random_state=SEED))]))
    Z["SVM_RBF"] = dict(subsample=SUBSAMPLE_SVM, note="kernel SVM on a stratified subsample",
        make=lambda: Pipeline(pre_scaled() + [("clf", SVC(
            C=2.0, gamma="scale", class_weight=cw, cache_size=1000,
            probability=True, random_state=SEED))]))
    return {k: v for k, v in Z.items() if k in MODELS_TO_RUN}


def _hgb(cw):
    """class_weight arrived in sklearn 1.4; fall back cleanly on older versions."""
    kw = dict(max_iter=400, learning_rate=0.06, max_leaf_nodes=31,
              l2_regularization=1.0, early_stopping=False, random_state=SEED)
    try:
        return HistGradientBoostingClassifier(class_weight=cw, **kw)
    except TypeError:
        print("  [note] sklearn < 1.4: HistGradientBoosting without class_weight")
        return HistGradientBoostingClassifier(**kw)


def subsample(X, y, n_max, seed=SEED):
    """Stratified cap on TRAINING size only; val/test are always scored in full."""
    if n_max is None or len(y) <= n_max:
        return X, y
    rng = np.random.default_rng(seed)
    pos, neg = np.flatnonzero(y == 1), np.flatnonzero(y == 0)
    n_pos = min(len(pos), n_max // 2)
    n_neg = min(len(neg), n_max - n_pos)
    idx = np.concatenate([rng.choice(pos, n_pos, replace=False),
                          rng.choice(neg, n_neg, replace=False)])
    rng.shuffle(idx)
    return X[idx], y[idx]


def proba(pipe, X):
    if hasattr(pipe, "predict_proba"):
        return np.asarray(pipe.predict_proba(X))[:, 1]
    s = np.asarray(pipe.decision_function(X), float)
    return 1.0 / (1.0 + np.exp(-np.clip(s, -30, 30)))


ZOO = build_models()
print(f"{len(ZOO)} models, class_weight={CLASS_WEIGHT}:")
for k, v in ZOO.items():
    print(f"  {k:22s} {'full data' if not v['subsample'] else f'subsample {v[chr(39)+chr(39)] if False else v[chr(115)+chr(117)+chr(98)+chr(115)+chr(97)+chr(109)+chr(112)+chr(108)+chr(101)]:,}':18s} {v['note']}")

## Cell 6 — The LOPO loop

For each model, for each of the 89 patients: fit on 76, tune the threshold on 12,
predict the held-out one, save.

Fold files live under `folds/{model}/{pid}.npz` so each model resumes independently.
A corrupt or truncated file is detected, deleted and refit rather than aborting the run
— the failure mode that cost hours in the deep-learning runs. Writes go through
`os.fsync` before the atomic rename, so a full disk raises immediately instead of
leaving a hollow file.

Runtime is dominated by the SVM. Expect roughly:

| Model | per fold | 89 folds |
|---|---|---|
| LogisticRegression | < 1 s | ~1 min |
| HistGradientBoosting | ~2 s | ~4 min |
| RandomForest | ~4 s | ~7 min |
| XGBoost | ~3 s | ~5 min |
| SVM_RBF | ~15 s | ~25 min |

In [ ]:
# =========================================================================
# CELL 6 — LOPO loop
# =========================================================================
def fold_path(model, pid):
    d = os.path.join(FOLD_DIR, model); os.makedirs(d, exist_ok=True)
    return os.path.join(d, f"{pid}.npz")


def load_fold(fp):
    if not os.path.exists(fp):
        return None
    try:
        with np.load(fp) as z:
            if not {"probs", "labels", "threshold", "val_f1"} <= set(z.files):
                raise ValueError("missing keys")
            r = dict(probs=z["probs"], labels=z["labels"],
                     threshold=float(z["threshold"]), val_f1=float(z["val_f1"]),
                     val_probs=z["val_probs"] if "val_probs" in z.files else None,
                     val_labels=z["val_labels"] if "val_labels" in z.files else None)
        if len(r["probs"]) != len(r["labels"]) or len(r["probs"]) == 0:
            raise ValueError("length mismatch")
        return r
    except Exception as e:
        print(f"    [warn] {os.path.basename(fp)} unreadable ({type(e).__name__}) -- refitting")
        os.remove(fp); return None


def save_fold(fp, r):
    tmp = fp + ".tmp.npz"
    with open(tmp, "wb") as fh:
        np.savez(fh, **{k: v for k, v in r.items() if v is not None})
        fh.flush(); os.fsync(fh.fileno())
    os.replace(tmp, fp)


def fit_fold(model_name, test_pid):
    spec = ZOO[model_name]
    tr_p, va_p, te_p = make_fold(test_pid)
    Xtr, ytr = assemble(tr_p); Xva, yva = assemble(va_p); Xte, yte = assemble(te_p)

    pipe = spec["make"]()
    Xs, ys = subsample(Xtr, ytr, spec["subsample"])
    if spec.get("pos_weight"):
        spw = float((ys == 0).sum() / max((ys == 1).sum(), 1)) if CLASS_WEIGHT else 1.0
        pipe.named_steps["clf"].set_params(scale_pos_weight=spw)
    pipe.fit(Xs, ys)

    pv = proba(pipe, Xva)                         # VALIDATION only
    thr, vf1 = best_f1_threshold(yva, pv)
    pt = proba(pipe, Xte)
    return dict(probs=pt.astype(np.float32), labels=yte.astype(np.int8),
                threshold=thr, val_f1=vf1,
                val_probs=pv.astype(np.float32), val_labels=yva.astype(np.int8)), pipe


def run_model(model_name, max_folds=MAX_FOLDS, resume=RESUME):
    pids = PIDS[:max_folds] if max_folds else PIDS
    rows, t0 = [], time.time()
    for i, pid in enumerate(pids, 1):
        fp = fold_path(model_name, pid)
        r = load_fold(fp) if resume else None
        tag = "cached"
        if r is None:
            r, pipe = fit_fold(model_name, pid)
            save_fold(fp, r); tag = "fit"
            if i == 1:
                try:
                    import joblib
                    joblib.dump(pipe, os.path.join(MODEL_DIR, f"{model_name}_fold0.joblib"))
                except Exception: pass
        m = patient_metrics(r["labels"], r["probs"], r["threshold"])
        m.update(patient=pid, model=model_name, val_f1=r["val_f1"], source=tag)
        rows.append(m)
        done = pd.DataFrame(rows)
        done.to_csv(os.path.join(CSV_DIR, f"per_patient_{model_name}.csv"), index=False)
        if i % 10 == 0 or i == len(pids):
            print(f"    [{model_name} {i}/{len(pids)}] running mean F1 "
                  f"{np.nanmean(done.f1):.4f}  AUC {np.nanmean(done.roc_auc):.4f}"
                  f"  ({time.time()-t0:.0f}s)")
    return pd.DataFrame(rows)


def run_all():
    out = {}
    for name in ZOO:
        done = len(glob.glob(os.path.join(FOLD_DIR, name, "*.npz")))
        print(f"\n{'='*76}\n  {name}  ({done} folds cached)\n{'='*76}")
        out[name] = run_model(name)
        gc.collect()
    return out

print(f"ready: {len(ZOO)} models x {len(PIDS)} folds = {len(ZOO)*len(PIDS)} fits")

## Cell 7 — Run

In [ ]:
# =========================================================================
# CELL 7 — RUN
# =========================================================================
T0 = time.time()
RESULTS = run_all()
print(f"\ncomplete in {(time.time()-T0)/60:.1f} min")
for k, v in RESULTS.items():
    print(f"  {k:22s} {len(v)} folds | mean F1 {np.nanmean(v.f1):.4f}")

## Cell 8 — Aggregate, compare, and test

### Two estimators, both reported

**LOPO mean (macro)** — compute the metric per patient, average the 89 values. Every
patient counts equally regardless of night length. This is the headline.

**Pooled (micro)** — concatenate all 89 patients, compute once. Long and
high-prevalence recordings dominate.

They differ because prevalence spans roughly 1% to 73% here. One caveat: pooled
accuracy/F1/sensitivity are sound (each patient is binarized at its own fold's
threshold first), but pooled **AUROC/PR-AUC/Brier/ECE** mix probabilities from 89
separately fitted models. Report the macro versions.

### Two free ensembles

Validation probabilities were stored per fold, so a soft vote and a logistic stacker
cost nothing extra. The stacker is fitted on **validation** probabilities and applied to
test — legitimate precisely because validation is not test.

### Paired tests

All models saw identical folds, so per-patient differences are paired. Wilcoxon
signed-rank (not a t-test — per-patient F1 is bounded, skewed, and hits a hard floor at
0), with Holm correction across the metric family and a rank-biserial effect size beside
every p-value.

In [ ]:
# =========================================================================
# CELL 8 — aggregation, ensembles, comparison tables, paired tests
# =========================================================================
def pooled_for(model, df):
    L, P, PR, VP, VL = [], [], [], [], []
    for pid in df.patient:
        r = load_fold(fold_path(model, pid))
        if r is None: continue
        L.append(r["labels"]); P.append(r["probs"])
        PR.append((r["probs"] >= r["threshold"]).astype(int))
        if r.get("val_probs") is not None:
            VP.append(r["val_probs"]); VL.append(r["val_labels"])
    lab, prob, pred = np.concatenate(L), np.concatenate(P), np.concatenate(PR)
    tn, fp, fn, tp = confusion_matrix(lab, pred, labels=[0, 1]).ravel()
    sens, spec = tp / (tp + fn), tn / (tn + fp)
    d = dict(accuracy=accuracy_score(lab, pred),
             precision=precision_score(lab, pred, zero_division=0),
             recall=sens, sensitivity=sens, specificity=spec,
             npv=tn / max(tn + fn, 1), f1=f1_score(lab, pred, zero_division=0),
             balanced_accuracy=(sens + spec) / 2, mcc=matthews_corrcoef(lab, pred),
             cohen_kappa=cohen_kappa_score(lab, pred),
             roc_auc=roc_auc_score(lab, prob), pr_auc=average_precision_score(lab, prob),
             brier=brier_score_loss(lab, np.clip(prob, 0, 1)),
             ece=expected_calibration_error(lab, prob))
    return d, lab, prob, pred, np.array([[tn, fp], [fn, tp]]), (VP, VL)


def summarize(df):
    rows = []
    for c in METRIC_COLS:
        v = df[c].astype(float).values; ok = v[np.isfinite(v)]
        rows.append(dict(metric=c, mean=ok.mean() if len(ok) else np.nan,
                         std=ok.std(ddof=1) if len(ok) > 1 else np.nan,
                         sem=ok.std(ddof=1)/np.sqrt(len(ok)) if len(ok) > 1 else np.nan,
                         median=np.median(ok) if len(ok) else np.nan,
                         min=ok.min() if len(ok) else np.nan,
                         max=ok.max() if len(ok) else np.nan,
                         n_valid=len(ok), n_undefined=len(v) - len(ok)))
    return pd.DataFrame(rows)


MDF, POOLED, CONF, RAW, EPH, VALP = {}, {}, {}, {}, {}, {}
for name in ZOO:
    fp = os.path.join(CSV_DIR, f"per_patient_{name}.csv")
    if not os.path.exists(fp): continue
    df = pd.read_csv(fp); MDF[name] = df
    pooled, lab, prob, pred, cm, vv = pooled_for(name, df)
    s = summarize(df); s["pooled"] = s.metric.map(pooled)
    s.to_csv(os.path.join(CSV_DIR, f"summary_{name}.csv"), index=False)
    POOLED[name], CONF[name] = pooled, cm
    RAW[name] = dict(labels=lab, probs=prob, pred=pred)
    VALP[name] = vv
    EPH[name] = eph_agreement(df)

DONE = list(MDF)

# ---- ensembles from the stored validation probabilities -----------------
if len(DONE) >= 3 and all(VALP[n][0] for n in DONE):
    order = PIDS[:MAX_FOLDS] if MAX_FOLDS else PIDS
    Pt = np.column_stack([RAW[n]["probs"] for n in DONE])
    Pv = np.column_stack([np.concatenate(VALP[n][0]) for n in DONE])
    yv = np.concatenate(VALP[DONE[0]][1]).astype(int)
    yt = RAW[DONE[0]]["labels"].astype(int)
    bounds, off = [], 0
    for pid in order:
        n = len(CACHE[pid]["y"]); bounds.append((pid, off, off + n)); off += n

    def _ens(name, pv, pt):
        thr, _ = best_f1_threshold(yv, pv)
        rows = [dict(patient=pid, model=name,
                     **patient_metrics(yt[s:e], pt[s:e], thr)) for pid, s, e in bounds]
        d = pd.DataFrame(rows); d.to_csv(os.path.join(CSV_DIR, f"per_patient_{name}.csv"),
                                         index=False)
        MDF[name] = d; EPH[name] = eph_agreement(d)
        pr = (pt >= thr).astype(int)
        tn, fp_, fn, tp = confusion_matrix(yt, pr, labels=[0, 1]).ravel()
        sens, spec = tp/(tp+fn), tn/(tn+fp_)
        POOLED[name] = dict(accuracy=accuracy_score(yt, pr),
            precision=precision_score(yt, pr, zero_division=0), recall=sens,
            sensitivity=sens, specificity=spec, npv=tn/max(tn+fn,1),
            f1=f1_score(yt, pr, zero_division=0), balanced_accuracy=(sens+spec)/2,
            mcc=matthews_corrcoef(yt, pr), cohen_kappa=cohen_kappa_score(yt, pr),
            roc_auc=roc_auc_score(yt, pt), pr_auc=average_precision_score(yt, pt),
            brier=brier_score_loss(yt, np.clip(pt, 0, 1)),
            ece=expected_calibration_error(yt, pt))
        CONF[name] = np.array([[tn, fp_], [fn, tp]])
        RAW[name] = dict(labels=yt, probs=pt, pred=pr)
        summarize(d).assign(pooled=lambda s: s.metric.map(POOLED[name])) \
            .to_csv(os.path.join(CSV_DIR, f"summary_{name}.csv"), index=False)

    _ens("SoftVote", Pv.mean(1), Pt.mean(1))
    meta = LogisticRegression(max_iter=1000).fit(Pv, yv)
    _ens("Stacked_LR", meta.predict_proba(Pv)[:, 1], meta.predict_proba(Pt)[:, 1])
    pd.DataFrame({"model": DONE, "stack_coef": meta.coef_[0]}) \
      .to_csv(os.path.join(CSV_DIR, "stacking_weights.csv"), index=False)
    DONE = list(MDF)

ALL = pd.concat([MDF[m].assign(model=m) for m in DONE], ignore_index=True)
ALL.to_csv(os.path.join(CSV_DIR, "per_patient_all_models.csv"), index=False)

# ---- headline + comparison tables ---------------------------------------
HEAD_COLS = METRIC_COLS
rows = []
for m in DONE:
    r = dict(model=m, n_folds=len(MDF[m]))
    for c in HEAD_COLS:
        r[c] = float(np.nanmean(MDF[m][c])); r[f"{c}_sd"] = float(np.nanstd(MDF[m][c], ddof=1))
        r[f"{c}_pooled"] = POOLED[m][c]
    r.update({k: v for k, v in EPH[m].items() if k != "n"})
    rows.append(r)
HEADLINE = pd.DataFrame(rows)
HEADLINE.to_csv(os.path.join(CSV_DIR, "model_headline.csv"), index=False)

REQ = [("accuracy","Accuracy"),("precision","Precision"),("recall","Recall"),
       ("sensitivity","Sensitivity"),("specificity","Specificity"),("npv","NPV"),
       ("f1","F1"),("balanced_accuracy","Balanced accuracy"),("mcc","MCC"),
       ("cohen_kappa","Cohen kappa"),("roc_auc","ROC AUC"),("pr_auc","PR AUC"),
       ("brier","Brier"),("ece","ECE")]
EPH_REQ = [("eph_pearson_r","Events/h Pearson r"),
           ("eph_r2_determination","Events/h R2 (determination)"),
           ("eph_r2_pearson","Events/h r2 (Pearson squared)"),
           ("eph_mae","Events/h MAE"),("eph_bias","Events/h bias"),
           ("severity_acc","Severity accuracy"),("severity_kappa_w","Severity kappa_w")]

rows = []
for k, lab in REQ:
    r = {"metric": lab, "key": k}
    for m in DONE:
        v = MDF[m][k].astype(float)
        r[m] = f"{np.nanmean(v):.4f} ± {np.nanstd(v, ddof=1):.4f}"
        r[f"{m} (pooled)"] = f"{POOLED[m][k]:.4f}"
    rows.append(r)
for k, lab in EPH_REQ:
    r = {"metric": lab, "key": k}
    for m in DONE:
        r[m] = f"{EPH[m][k]:.4f}"; r[f"{m} (pooled)"] = ""
    rows.append(r)
COMPARISON = pd.DataFrame(rows)
COMPARISON.to_csv(os.path.join(CSV_DIR, "metric_comparison.csv"), index=False)

WIDE = pd.DataFrame({"metric": [l for _, l in REQ + EPH_REQ],
                     "key": [k for k, _ in REQ + EPH_REQ]})
for m in DONE:
    WIDE[m] = ([float(np.nanmean(MDF[m][k])) for k, _ in REQ]
               + [float(EPH[m][k]) for k, _ in EPH_REQ])
def _best(row):
    v = row[DONE].astype(float)
    if not np.isfinite(v).any(): return "n/a"
    if row["key"] in ("brier", "ece", "eph_mae"): return v.idxmin()
    if row["key"] == "eph_bias": return v.abs().idxmin()
    return v.idxmax()
WIDE["best_model"] = WIDE.apply(_best, axis=1)
WIDE.to_csv(os.path.join(CSV_DIR, "metric_comparison_wide.csv"), index=False)
pd.DataFrame([{"model": m, **EPH[m]} for m in DONE]) \
  .to_csv(os.path.join(CSV_DIR, "eph_agreement.csv"), index=False)

# ---- severity strata + per-band severity accuracy ------------------------
_ord = {n: i for i, n in enumerate(SEVERITY_NAMES)}
STRATA = (ALL.groupby(["model", "severity"])
          .agg(n_patients=("patient","count"), prevalence=("prevalence","mean"),
               accuracy=("accuracy","mean"), f1=("f1","mean"),
               precision=("precision","mean"), sensitivity=("sensitivity","mean"),
               specificity=("specificity","mean"), roc_auc=("roc_auc","mean"),
               pr_auc=("pr_auc","mean"), cohen_kappa=("cohen_kappa","mean"),
               mcc=("mcc","mean")).reset_index()
          .sort_values(["model","severity"],
                       key=lambda s: s.map(_ord) if s.name == "severity" else s))
STRATA.to_csv(os.path.join(CSV_DIR, "severity_strata_by_model.csv"), index=False)

sev_rows, sev_cm = [], {}
for m in DONE:
    d = MDF[m]
    ts = np.array([severity_of(v) for v in d.true_eph])
    ps = np.array([severity_of(v) for v in d.pred_eph])
    sev_cm[m] = confusion_matrix(ts, ps, labels=range(len(SEVERITY_NAMES)))
    for b, nm in enumerate(SEVERITY_NAMES):
        sel = ts == b
        sev_rows.append(dict(model=m, severity=nm, n_patients=int(sel.sum()),
            severity_recall=float((ps[sel] == b).mean()) if sel.any() else np.nan,
            severity_precision=float((ts[ps == b] == b).mean()) if (ps == b).any() else np.nan,
            mean_true_eph=float(d.true_eph[sel].mean()) if sel.any() else np.nan,
            mean_pred_eph=float(d.pred_eph[sel].mean()) if sel.any() else np.nan,
            mean_abs_err=float(np.abs(d.pred_eph - d.true_eph)[sel].mean()) if sel.any() else np.nan))
SEV_ACC = pd.DataFrame(sev_rows)
SEV_ACC.to_csv(os.path.join(CSV_DIR, "severity_accuracy_by_band.csv"), index=False)

# ---- pooled confusion matrices ------------------------------------------
rows = []
for m in DONE:
    (tn, fp_), (fn, tp) = CONF[m]
    n = tn + fp_ + fn + tp
    rows.append(dict(model=m, n_epochs=int(n), tn=int(tn), fp=int(fp_),
                     fn=int(fn), tp=int(tp), sensitivity=tp/(tp+fn),
                     specificity=tn/(tn+fp_), precision=tp/max(tp+fp_,1),
                     npv=tn/max(tn+fn,1), accuracy=(tp+tn)/n))
CM_DF = pd.DataFrame(rows)
CM_DF.to_csv(os.path.join(CSV_DIR, "confusion_matrices.csv"), index=False)

# ---- paired Wilcoxon vs the baseline model ------------------------------
def holm(p):
    p = np.asarray(p, float); m = len(p); adj = np.empty(m); run = 0.0
    for rank, idx in enumerate(np.argsort(p)):
        run = max(run, (m - rank) * p[idx]); adj[idx] = min(run, 1.0)
    return adj

LOWER_BETTER = {"brier", "ece"}
TEST_METRICS = [c for c in METRIC_COLS]
rows = []
if BASELINE_MODEL in MDF:
    base = MDF[BASELINE_MODEL]
    for m in DONE:
        if m == BASELINE_MODEL: continue
        mg = base[["patient"] + TEST_METRICS].merge(
            MDF[m][["patient"] + TEST_METRICS], on="patient", suffixes=("_b", "_m"))
        for c in TEST_METRICS:
            b = mg[f"{c}_b"].astype(float).values; v = mg[f"{c}_m"].astype(float).values
            ok = np.isfinite(b) & np.isfinite(v); b, v = b[ok], v[ok]
            d = (b - v) if c in LOWER_BETTER else (v - b)     # + = improvement
            nz = d[d != 0]
            if len(nz) < 5:
                stat = p = rb = np.nan
            else:
                stat, p = wilcoxon(v, b, zero_method="wilcox")
                rb = (nz > 0).mean() - (nz < 0).mean()
            rows.append(dict(model=m, metric=c, n_paired=len(b),
                             baseline_mean=b.mean(), model_mean=v.mean(),
                             median_delta=np.median(d) if len(d) else np.nan,
                             n_improved=int((d > 0).sum()), n_degraded=int((d < 0).sum()),
                             rank_biserial=rb, p_raw=p))
TESTS = pd.DataFrame(rows)
if len(TESTS):
    TESTS["p_holm"] = holm(TESTS.p_raw.fillna(1.0).values)
    TESTS["sig"] = np.where(TESTS.p_holm < 0.001, "***",
                    np.where(TESTS.p_holm < 0.01, "**",
                      np.where(TESTS.p_holm < 0.05, "*", "ns")))
    TESTS.to_csv(os.path.join(CSV_DIR, f"paired_tests_vs_{BASELINE_MODEL}.csv"), index=False)

print("=" * 120)
print(f"  33 AASM FEATURES + CLASSICAL ML — {len(DONE)} models x {len(MDF[DONE[0]])} LOPO folds")
print("=" * 120)
print(COMPARISON[["metric"] + DONE].to_string(index=False))
print("\n---- BEST MODEL PER METRIC ----")
print(WIDE[["metric"] + DONE + ["best_model"]].to_string(index=False,
      float_format=lambda v: f"{v:9.4f}"))
print(f"\nwins: {WIDE[WIDE.best_model != 'n/a'].best_model.value_counts().to_dict()}")
print("\n---- SEVERITY ACCURACY BY BAND ----")
print(SEV_ACC.to_string(index=False, float_format=lambda v: f"{v:9.4f}"))
print("\n---- POOLED CONFUSION ----")
print(CM_DF.to_string(index=False, float_format=lambda v: f"{v:9.4f}"))
if len(TESTS):
    print(f"\n---- PAIRED WILCOXON vs {BASELINE_MODEL} (Holm-corrected) ----")
    print(TESTS[["model","metric","baseline_mean","model_mean","median_delta",
                 "n_improved","n_degraded","rank_biserial","p_holm","sig"]]
          .to_string(index=False, float_format=lambda v: f"{v:9.4f}"))

## Cell 9 — Figures

Eleven plots in `figures/`, at 150 dpi — the same set the deep-learning notebooks
produce, so the two studies can be placed side by side in a paper.

1. **Metric bars** with between-patient SD. Overlapping error bars mean the difference
   is not real however different the point estimates look.
2. **Per-patient distributions** (F1, AUROC, MCC, κ). The spread matters as much as the
   mean: a model with a higher mean and a longer bottom tail is worse for screening.
3. **Paired slope plot** — one line per patient from the baseline to each model. Crossing
   lines mean the models disagree per patient rather than one dominating.
4. **Pooled ROC** and **5. pooled PR**, with the prevalence chance floor marked.
6. **Reliability diagrams** with ECE in the legend.
7. **Events per hour, predicted vs actual** — one panel per model, with the identity
   line, the fitted line, R², r and MAE, and the severity band boundaries drawn as a
   grid. Dots in the wrong grid cell are patients whose severity band was misassigned.
8. **Bland–Altman**, shared axes so the limits of agreement are directly comparable.
9. **Severity confusion matrices** — predicted band against true band, row-normalized so
   the diagonal reads as per-band recall.
10. **Per-band severity accuracy** grouped by model.
11. **F1 and AUROC against prevalence** — the operating-point figure. A steep F1 slope
    beside a flat AUROC slope means discrimination is uniform and the threshold is what
    fails.

In [ ]:
# =========================================================================
# CELL 9 — figures
# =========================================================================
def _save(fig, name):
    fig.tight_layout()
    fig.savefig(os.path.join(FIG_DIR, f"{name}.png"), dpi=150, bbox_inches="tight")
    plt.show()

def _box(ax, data, labels):
    try:    return ax.boxplot(data, showmeans=True, patch_artist=True, tick_labels=labels)
    except TypeError: return ax.boxplot(data, showmeans=True, patch_artist=True, labels=labels)

C = [MODEL_COLOR.get(m, "#888888") for m in DONE]

# --- 1 metric bars -------------------------------------------------------
bars = ["accuracy","precision","sensitivity","specificity","f1",
        "balanced_accuracy","mcc","cohen_kappa","roc_auc","pr_auc"]
fig, ax = plt.subplots(figsize=(15, 5.5))
x = np.arange(len(bars)); w = 0.8 / len(DONE)
for k, m in enumerate(DONE):
    mu = [np.nanmean(MDF[m][b]) for b in bars]
    sd = [np.nanstd(MDF[m][b], ddof=1) for b in bars]
    ax.bar(x + k*w - 0.4 + w/2, mu, w, yerr=sd, capsize=2, label=m,
           color=MODEL_COLOR.get(m, "#888"), alpha=.9, error_kw=dict(lw=.7, alpha=.6))
ax.set_xticks(x); ax.set_xticklabels(bars, rotation=25, ha="right")
ax.set_ylabel("LOPO mean ± SD"); ax.set_ylim(0, 1.05)
ax.set_title(f"33 AASM features — {len(MDF[DONE[0]])} leave-one-patient-out folds")
ax.legend(ncol=min(4, len(DONE)), fontsize=8); ax.grid(axis="y", alpha=.3)
_save(fig, "01_metric_bars")

# --- 2 distributions -----------------------------------------------------
fig, axes = plt.subplots(1, 4, figsize=(18, 4.8))
for ax, c in zip(axes, ["f1", "roc_auc", "mcc", "cohen_kappa"]):
    bp = _box(ax, [MDF[m][c].dropna().values for m in DONE], DONE)
    for patch, col in zip(bp["boxes"], C): patch.set_facecolor(col); patch.set_alpha(.55)
    ax.set_title(c); ax.grid(axis="y", alpha=.3); ax.tick_params(axis="x", rotation=30)
fig.suptitle("Per-patient distribution across folds")
_save(fig, "02_distributions")

# --- 3 paired slopes -----------------------------------------------------
others = [m for m in DONE if m != BASELINE_MODEL]
if BASELINE_MODEL in MDF and others:
    fig, axes = plt.subplots(1, len(others), figsize=(3.6*len(others), 5), sharey=True)
    axes = np.atleast_1d(axes)
    for ax, m in zip(axes, others):
        mg = MDF[BASELINE_MODEL][["patient","f1"]].merge(
            MDF[m][["patient","f1"]], on="patient", suffixes=("_b","_m")).dropna()
        for _, r in mg.iterrows():
            ax.plot([0,1],[r.f1_b, r.f1_m], lw=.6, alpha=.5,
                    c="#59a14f" if r.f1_m >= r.f1_b else "#d1495b")
        ax.plot([0,1],[mg.f1_b.mean(), mg.f1_m.mean()], "o-", lw=2.4, c="k",
                label=f"{mg.f1_b.mean():.3f} → {mg.f1_m.mean():.3f}")
        ax.set_xticks([0,1]); ax.set_xticklabels([BASELINE_MODEL, m], rotation=12, fontsize=8)
        ax.set_title(f"{int((mg.f1_m>mg.f1_b).sum())} better / "
                     f"{int((mg.f1_m<mg.f1_b).sum())} worse", fontsize=9)
        ax.legend(fontsize=7); ax.grid(axis="y", alpha=.3)
    axes[0].set_ylabel("per-patient F1")
    fig.suptitle(f"Paired per-patient F1 vs {BASELINE_MODEL}")
    _save(fig, "03_paired_slopes")

# --- 4/5 ROC and PR ------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for m in DONE:
    r = RAW[m]
    fpr, tpr, _ = roc_curve(r["labels"], r["probs"])
    axes[0].plot(fpr, tpr, lw=1.6, c=MODEL_COLOR.get(m,"#888"),
                 label=f"{m} ({POOLED[m]['roc_auc']:.4f})")
    pr, rc, _ = precision_recall_curve(r["labels"], r["probs"])
    axes[1].plot(rc, pr, lw=1.6, c=MODEL_COLOR.get(m,"#888"),
                 label=f"{m} ({POOLED[m]['pr_auc']:.4f})")
axes[0].plot([0,1],[0,1],"k--",lw=.8)
axes[0].set_xlabel("false positive rate"); axes[0].set_ylabel("true positive rate")
axes[0].set_title("Pooled ROC"); axes[0].legend(fontsize=8); axes[0].grid(alpha=.3)
prev = RAW[DONE[0]]["labels"].mean()
axes[1].axhline(prev, ls="--", c="k", lw=.8, label=f"chance = {prev:.3f}")
axes[1].set_xlabel("recall"); axes[1].set_ylabel("precision")
axes[1].set_title("Pooled precision–recall"); axes[1].legend(fontsize=8); axes[1].grid(alpha=.3)
_save(fig, "04_roc_pr")

# --- 6 reliability -------------------------------------------------------
fig, ax = plt.subplots(figsize=(6.5, 6))
edges = np.linspace(0, 1, N_CALIB_BINS + 1)
for m in DONE:
    r = RAW[m]; xs, ys = [], []
    for i in range(N_CALIB_BINS):
        hi = r["probs"] <= edges[i+1] if i == N_CALIB_BINS-1 else r["probs"] < edges[i+1]
        sel = (r["probs"] >= edges[i]) & hi
        if sel.any(): xs.append(r["probs"][sel].mean()); ys.append(r["labels"][sel].mean())
    ax.plot(xs, ys, "o-", lw=1.4, ms=4, c=MODEL_COLOR.get(m,"#888"),
            label=f"{m} (ECE {POOLED[m]['ece']:.4f})")
ax.plot([0,1],[0,1],"k--",lw=.8,label="perfect")
ax.set_xlabel("mean predicted probability"); ax.set_ylabel("observed frequency")
ax.set_title("Reliability"); ax.legend(fontsize=8); ax.grid(alpha=.3)
_save(fig, "05_calibration")

# --- 7 events per hour: predicted vs actual ------------------------------
lim = max(ALL.true_eph.max(), ALL.pred_eph.max()) * 1.08
fig, axes = plt.subplots(1, len(DONE), figsize=(4.1*len(DONE), 4.5), sharex=True, sharey=True)
axes = np.atleast_1d(axes)
for ax, m in zip(axes, DONE):
    d = MDF[m]
    ax.scatter(d.true_eph, d.pred_eph, s=30, c=MODEL_COLOR.get(m,"#888"), alpha=.8)
    ax.plot([0,lim],[0,lim],"k--",lw=1,label="identity")
    z = np.polyfit(d.true_eph, d.pred_eph, 1)
    ax.plot([0,lim], np.polyval(z,[0,lim]), "-", c="grey", lw=1.2, label="fit")
    for b in EPH_BINS: ax.axvline(b, c="#ccc", lw=.6); ax.axhline(b, c="#ccc", lw=.6)
    ax.set_title(f"{m}\nR2={EPH[m]['eph_r2_determination']:.3f}  "
                 f"r={EPH[m]['eph_pearson_r']:.3f}  MAE={EPH[m]['eph_mae']:.2f}/h", fontsize=9)
    ax.set_xlabel("true events/h"); ax.set_xlim(0,lim); ax.set_ylim(0,lim)
    ax.legend(fontsize=7); ax.grid(alpha=.3)
axes[0].set_ylabel("predicted events/h")
fig.suptitle("Events per hour — predicted vs actual (grid lines = severity bands)")
_save(fig, "06_events_per_hour")

# --- 8 Bland-Altman ------------------------------------------------------
fig, axes = plt.subplots(1, len(DONE), figsize=(4.1*len(DONE), 4.5), sharey=True)
axes = np.atleast_1d(axes)
for ax, m in zip(axes, DONE):
    d = MDF[m]; diff = d.pred_eph - d.true_eph; mn = (d.pred_eph + d.true_eph)/2
    bias, sd = diff.mean(), diff.std(ddof=1)
    ax.scatter(mn, diff, s=30, c=MODEL_COLOR.get(m,"#888"), alpha=.8)
    ax.axhline(bias, c="k", lw=1.1); ax.axhline(0, c="grey", lw=.7)
    ax.axhline(bias+1.96*sd, c="r", ls="--", lw=1); ax.axhline(bias-1.96*sd, c="r", ls="--", lw=1)
    ax.set_title(f"{m}\nbias {bias:+.2f}  LoA [{bias-1.96*sd:+.2f}, {bias+1.96*sd:+.2f}]",
                 fontsize=9)
    ax.set_xlabel("mean of true and predicted"); ax.grid(alpha=.3)
axes[0].set_ylabel("predicted − true")
fig.suptitle("Events per hour — Bland–Altman")
_save(fig, "07_bland_altman")

# --- 9 severity confusion matrices ---------------------------------------
present = [i for i, n in enumerate(SEVERITY_NAMES)
           if any((sev_cm[m][i].sum() + sev_cm[m][:, i].sum()) > 0 for m in sev_cm)]
if present:
    fig, axes = plt.subplots(1, len(sev_cm), figsize=(3.7*len(sev_cm), 3.8))
    axes = np.atleast_1d(axes)
    for ax, m in zip(axes, sev_cm):
        cm = sev_cm[m][np.ix_(present, present)].astype(float)
        rn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
        ax.imshow(rn, cmap="Blues", vmin=0, vmax=1)
        for i in range(len(present)):
            for j in range(len(present)):
                ax.text(j, i, f"{rn[i,j]:.2f}\n({int(cm[i,j])})", ha="center",
                        va="center", fontsize=8,
                        color="white" if rn[i,j] > .55 else "black")
        lb = [SEVERITY_NAMES[i] for i in present]
        ax.set_xticks(range(len(present))); ax.set_xticklabels(lb, rotation=30, fontsize=8)
        ax.set_yticks(range(len(present))); ax.set_yticklabels(lb, fontsize=8)
        ax.set_title(m, fontsize=9); ax.set_xlabel("predicted band")
    axes[0].set_ylabel("true band")
    fig.suptitle("Severity confusion (row-normalized: diagonal = per-band recall)")
    _save(fig, "08_severity_confusion")

# --- 10 per-band severity accuracy ---------------------------------------
bands = [b for b in SEVERITY_NAMES if (SEV_ACC.severity == b).any()
         and SEV_ACC[SEV_ACC.severity == b].n_patients.max() > 0]
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(bands)); w = 0.8/len(DONE)
for k, m in enumerate(DONE):
    vals = [SEV_ACC[(SEV_ACC.model==m)&(SEV_ACC.severity==b)].severity_recall.mean()
            for b in bands]
    ax.bar(x + k*w - 0.4 + w/2, vals, w, label=m, color=MODEL_COLOR.get(m,"#888"), alpha=.9)
for i, b in enumerate(bands):
    n = int(SEV_ACC[SEV_ACC.severity==b].n_patients.max())
    ax.text(i, 0.02, f"n={n}", ha="center", fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(bands); ax.set_ylim(0, 1.05)
ax.set_ylabel("severity recall (correctly banded)"); ax.legend(fontsize=8)
ax.set_title("Per-band severity accuracy"); ax.grid(axis="y", alpha=.3)
_save(fig, "09_severity_accuracy")

# --- 11 the operating-point figure ---------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharex=True)
for ax, c, t in ((axes[0], "f1", "per-patient F1"), (axes[1], "roc_auc", "per-patient AUROC")):
    for m in DONE:
        ok = MDF[m][[c, "prevalence"]].dropna()
        ax.scatter(ok.prevalence, ok[c], s=22, alpha=.6, c=MODEL_COLOR.get(m,"#888"))
        if len(ok) > 2:
            z = np.polyfit(ok.prevalence, ok[c], 1)
            xs = np.linspace(ok.prevalence.min(), ok.prevalence.max(), 20)
            r = np.corrcoef(ok.prevalence, ok[c])[0,1]
            ax.plot(xs, np.polyval(z, xs), lw=1.6, c=MODEL_COLOR.get(m,"#888"),
                    label=f"{m}  r={r:+.3f}")
    ax.set_xlabel("apnea prevalence"); ax.set_ylabel(c); ax.set_title(t)
    ax.set_ylim(0, 1.02); ax.legend(fontsize=7); ax.grid(alpha=.3)
fig.suptitle("Steep F1 slope with flat AUROC = operating-point failure, not discrimination failure")
_save(fig, "10_operating_point")

print(f"figures -> {FIG_DIR}")

## Cell 10 — Feature importance and report

Permutation importance is computed on **held-out** data, so it reflects what each feature
contributed to generalization rather than to fitting. Its known weakness is correlated
features — `flow_amp_ratio` and `flow_min_ratio` carry overlapping information, so
shuffling either alone understates both. The **block-level** roll-up (respiratory /
SpO₂ / pulse / paradox) is therefore the more trustworthy reading, and it answers the
question worth asking: which part of the AASM rule is actually carrying the model?

In [ ]:
# =========================================================================
# CELL 10 — feature importance and final report
# =========================================================================
def feat_block(n):
    b = n.split("__")[0]
    if b.startswith("spo2"): return "SpO2"
    if b.startswith("pulse"): return "pulse"
    if b.startswith(("paradox", "thorax")): return "paradox"
    return "respiratory"

TREE = next((m for m in ("XGBoost","RandomForest","HistGradientBoosting") if m in ZOO), None)
IMP = None
if TREE:
    try:
        import joblib
        pipe = joblib.load(os.path.join(MODEL_DIR, f"{TREE}_fold0.joblib"))
        Xte, yte = assemble(make_fold(PIDS[0])[2])
        rng = np.random.default_rng(SEED)
        idx = rng.choice(len(yte), min(6000, len(yte)), replace=False)
        pi = permutation_importance(pipe, Xte[idx], yte[idx], n_repeats=5,
                                    random_state=SEED, n_jobs=N_JOBS, scoring="roc_auc")
        IMP = pd.DataFrame({"feature": FEATURE_NAMES, "perm_drop": pi.importances_mean,
                            "perm_std": pi.importances_std})
        IMP["block"] = IMP.feature.map(feat_block)
        IMP = IMP.sort_values("perm_drop", ascending=False)
        IMP.to_csv(os.path.join(CSV_DIR, f"feature_importance_{TREE}.csv"), index=False)
        blk = IMP.groupby("block").perm_drop.agg(["sum","mean","max","count"])
        blk["share_%"] = 100*blk["sum"]/max(blk["sum"].sum(), 1e-9)
        blk.sort_values("share_%", ascending=False).to_csv(
            os.path.join(CSV_DIR, "feature_importance_by_block.csv"))
        print(f"---- PERMUTATION IMPORTANCE ({TREE}, AUROC drop) ----")
        print(IMP.head(15).to_string(index=False, float_format=lambda v: f"{v:9.5f}"))
        print("\n---- BY AASM BLOCK ----")
        print(blk.sort_values("share_%", ascending=False)
              .to_string(float_format=lambda v: f"{v:9.4f}"))

        fig, axes = plt.subplots(1, 2, figsize=(15, 6))
        top = IMP.head(20).iloc[::-1]
        axes[0].barh(range(len(top)), top.perm_drop, xerr=top.perm_std,
                     color="#4c78a8", error_kw=dict(lw=.6))
        axes[0].set_yticks(range(len(top))); axes[0].set_yticklabels(top.feature, fontsize=7)
        axes[0].set_title(f"Top 20 features ({TREE})"); axes[0].grid(axis="x", alpha=.3)
        bs = blk.sort_values("share_%")
        axes[1].barh(range(len(bs)), bs["share_%"], color="#d1495b")
        axes[1].set_yticks(range(len(bs))); axes[1].set_yticklabels(bs.index, fontsize=9)
        axes[1].set_xlabel("% of total importance"); axes[1].set_title("By AASM block")
        axes[1].grid(axis="x", alpha=.3)
        _save(fig, "11_feature_importance")
    except Exception as e:
        print("[warn] permutation importance unavailable:", e)

# ---- report --------------------------------------------------------------
print("\n" + "=" * 100)
print(f"  REPORT — 33 AASM features, {len(DONE)} models, {len(MDF[DONE[0]])} LOPO folds")
print(f"  class_weight={CLASS_WEIGHT} (from BEST_LOSS_ARM='{BEST_LOSS_ARM}')")
print("=" * 100)
PAPER = ["f1","sensitivity","specificity","precision","accuracy","mcc",
         "cohen_kappa","roc_auc","pr_auc","ece"]
for c in PAPER:
    line = f"  {c:20s}"
    for m in DONE:
        v = MDF[m][c].astype(float)
        line += f" | {m[:9]:>9s} {np.nanmean(v):.4f}±{np.nanstd(v, ddof=1):.3f}"
    print(line)
print("\n  events per hour")
for k, lab in [("eph_pearson_r","Pearson r"),("eph_r2_determination","R2"),
               ("eph_mae","MAE /h"),("eph_bias","bias /h"),
               ("severity_acc","severity acc"),("severity_kappa_w","severity kappa_w")]:
    print(f"  {lab:20s}" + "".join(f" | {m[:9]:>9s} {EPH[m][k]:+.4f}" for m in DONE))

print("\n  pooled F1 / macro F1 (gap = long, high-prevalence recordings dominating)")
for m in DONE:
    print(f"    {m:22s} pooled {POOLED[m]['f1']:.4f}   macro {np.nanmean(MDF[m]['f1']):.4f}"
          f"   gap {POOLED[m]['f1']-np.nanmean(MDF[m]['f1']):+.4f}")

auc_span = max(np.nanmean(MDF[m].roc_auc) for m in DONE) - min(np.nanmean(MDF[m].roc_auc) for m in DONE)
f1_span  = max(np.nanmean(MDF[m].f1) for m in DONE) - min(np.nanmean(MDF[m].f1) for m in DONE)
print(f"\n  AUROC span across models {auc_span:.4f} | F1 span {f1_span:.4f}")

print(f"\n  output: {OUT_DIR}")
for lab, d in (("csv", CSV_DIR), ("figures", FIG_DIR), ("models", MODEL_DIR)):
    fs = sorted(os.listdir(d))
    print(f"    {lab:8s} {len(fs):3d} files")
    if lab == "csv":
        for f in fs: print(f"        {f}")